# Air Quality prediction model search - linear model

Loop and grid search for the optimal air quality model.

In [2]:
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_val_predict

In [3]:
data_folder = "/Users/martinfleis/Library/CloudStorage/OneDrive-UniverzitaKarlova,Přírodovědeckáfakulta/demoland_data"

Load the data

In [4]:
data = gpd.read_parquet(f"{data_folder}/processed/interpolated/all_oa.parquet")

Filter only explanatory variables.

In [5]:
exvars = data.drop(
    columns=[
        "geo_code",
        "geometry",
        "air_quality_index",
        "house_price_index",
        "jobs_accessibility_index",
        "greenspace_accessibility_index",
    ]
)

Specify grid search parameters. We can limit the options based on previous exploration.

In [6]:
parameters = {"learning_rate": (0.05, 0.1), "max_iter": [500], "max_bins": (64, 128)}
parameters = {}

Define the simple weights matrices.

In [7]:
queen = libpysal.weights.Queen.from_dataframe(data)
weights = {
    "queen": queen,
    "queen2": libpysal.weights.higher_order(queen, k=2, lower_order=True),
    "queen3": libpysal.weights.higher_order(queen, k=3, lower_order=True),
    "queen4": libpysal.weights.higher_order(queen, k=4, lower_order=True),
    "queen5": libpysal.weights.higher_order(queen, k=5, lower_order=True),
    "500m": libpysal.weights.DistanceBand.from_dataframe(data, 500),
    "1000m": libpysal.weights.DistanceBand.from_dataframe(data, 1000),
    "2000m": libpysal.weights.DistanceBand.from_dataframe(data, 2000),
}

/var/folders/n0/xbnn2f2d0ybbp2c_bf_9ztzw0000gn/T/ipykernel_67422/2015151665.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  queen = libpysal.weights.Queen.from_dataframe(data)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/contiguity.py:354: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/util.py:544: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  return W(neighbors=d, **kwargs)
/Users/martinfleis/dev/demoland-project/.pixi/envs/default/lib/python3.15/site-packages/libpysal/weights/util.py:819: UserWarning: The weights matrix is not fully connected: 
 There are 110 disconne

Use Grid Search CV to find the best model for each weights option.

In [8]:
meta = {}
for name, W in weights.items():
    W.transform = "r"
    exvars = data.drop(
        columns=[
            "geo_code",
            "geometry",
            "air_quality_index",
            "house_price_index",
            "jobs_accessibility_index",
            "greenspace_accessibility_index",
        ]
    )
    for col in exvars.columns.copy():
        exvars[f"{col}_lag"] = libpysal.weights.spatial_lag.lag_spatial(W, exvars[col])
    regressor_lag = LinearRegression()
    est_lag = GridSearchCV(regressor_lag, parameters, verbose=1)
    est_lag.fit(exvars, data.air_quality_index)
    meta[name] = {"score": est_lag.best_score_}
    y_pred_lag = cross_val_predict(
        est_lag.best_estimator_, exvars, data.air_quality_index, cv=5
    )
    pred_lag = pd.Series(y_pred_lag, index=data.index)
    residuals_lag = data.air_quality_index - pred_lag
    meta[name]["mse"] = mean_squared_error(data.air_quality_index, pred_lag)
    meta[name]["me"] = residuals_lag.abs().mean()
    meta[name]["prediction"] = pred_lag
    meta[name]["residuals"] = residuals_lag
    meta[name]["model"] = est_lag.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
('WARNING: ', 47, ' is an island (no neighbors)')
('WARNING: ', 71, ' is an island (no neighbors)')
('WARNING: ', 72, ' is an island (no neighbors)')
('WARNING: ', 89, ' is an island (no neighbors)')
('WARNING: ', 263, ' is an island (no neighbors)')
('WARNING: ', 361, ' is an island (no neighbors)')
('WARNING: ', 364, ' is an island (no neighbors)')
('WARNING: ', 375, ' is an island (no neighbors)')
('WARNING: ', 376, ' is an island (no neighbors)')
('WARNING: ', 377, ' is an island (no neighbors)')
('WARNING: ', 378, ' is an island (no neighbors)')
('WARNING: ', 541, ' is an island (no neighbors)')
('WARNING: ', 642, ' is an island (no neighbors)')
('WARNING: ', 983, ' is an island (no neighbor

Add combined weights on top.

In [9]:
combined_weigts = {
    "queen500m": libpysal.weights.w_union(weights["queen"], weights["500m"]),
    "queen1000m": libpysal.weights.w_union(weights["queen"], weights["1000m"]),
    "queen2000m": libpysal.weights.w_union(weights["queen"], weights["2000m"]),
}

Find models.

In [10]:
for name, W in combined_weigts.items():
    W.transform = "r"
    exvars = data.drop(
        columns=[
            "geo_code",
            "geometry",
            "air_quality_index",
            "house_price_index",
            "jobs_accessibility_index",
            "greenspace_accessibility_index",
        ]
    )
    for col in exvars.columns.copy():
        exvars[f"{col}_lag"] = libpysal.weights.spatial_lag.lag_spatial(W, exvars[col])
    regressor_lag = LinearRegression()
    est_lag = GridSearchCV(regressor_lag, parameters, verbose=1)
    est_lag.fit(exvars, data.air_quality_index)
    meta[name] = {"score": est_lag.best_score_}
    y_pred_lag = cross_val_predict(
        est_lag.best_estimator_, exvars, data.air_quality_index, cv=5
    )
    pred_lag = pd.Series(y_pred_lag, index=data.index)
    residuals_lag = data.air_quality_index - pred_lag
    meta[name]["mse"] = mean_squared_error(data.air_quality_index, pred_lag)
    meta[name]["me"] = residuals_lag.abs().mean()
    meta[name]["prediction"] = pred_lag
    meta[name]["residuals"] = residuals_lag
    meta[name]["model"] = est_lag.best_estimator_

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits


Save evaluation metrics as series.

In [11]:
mse = pd.Series([vals["mse"] for vals in meta.values()], index=meta.keys())
me = pd.Series([vals["me"] for vals in meta.values()], index=meta.keys())
score = pd.Series([vals["score"] for vals in meta.values()], index=meta.keys())

Sort according to MSE. Lower is better.

In [12]:
mse.sort_values()

queen2000m    0.652185
2000m         0.667587
queen5        0.749675
queen4        0.819483
queen3        0.960486
queen2        1.121166
queen1000m    1.140017
1000m         1.203097
queen500m     1.311564
queen         1.318673
500m          1.508573
dtype: float64

Sort according to ME. Lower is better.

In [13]:
me.sort_values()

queen2000m    0.630415
2000m         0.634449
queen5        0.671606
queen4        0.710151
queen3        0.772568
queen1000m    0.833097
queen2        0.839639
1000m         0.851907
queen         0.897791
queen500m     0.898503
500m          0.951924
dtype: float64

Sort according to R2. Higher is better.

In [14]:
score.sort_values()

500m          0.169478
queen500m     0.263369
queen         0.266544
1000m         0.327998
queen1000m    0.352741
queen2        0.366024
queen3        0.458288
queen4        0.536010
queen5        0.586991
2000m         0.594018
queen2000m    0.600105
dtype: float64